# ROBERT Run Wrapper and Output Archiver

Use this notebook to run ROBERT (optional) and archive `CURATE`, `GENERATE`, `VERIFY`, and `PREDICT` into a unique, timestamped folder per run.

This prevents new runs from overwriting previous generic outputs.

## Cell 2 Guide: Choose Dataset and ROBERT Options

This is the main setup cell for each run.

What to edit here before running:
- `DATASET_RELATIVE`: the dataset to use. You can provide either:
  - a path relative to project root (example: `Databases/Regression/Hvapor.csv`), or
  - just a filename (example: `vapor.csv`) and the notebook will search inside `Databases/`.
- `ROBERT_OPTIONS`: the command options that become `--key value` flags.
- `RUN_ROBERT`: set to `True` when you are ready to execute ROBERT.

You should not need to edit command text manually in other cells.

In [25]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import subprocess
from difflib import get_close_matches

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "robert").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not infer project root from notebook working directory. "
        "Expected a folder containing AGENTS.md and robert/."
    )


def dedupe_paths_case_insensitive(paths: list[Path]) -> list[Path]:
    bucket = {}
    for p in paths:
        key = str(p.resolve()).lower()
        if key not in bucket:
            bucket[key] = p.resolve()
    return list(bucket.values())


def resolve_dataset_path(project_root: Path, dataset_input: str) -> Path:
    candidate = Path(dataset_input)

    # 1) Absolute or direct relative path from project root
    if candidate.is_absolute() and candidate.exists():
        return candidate.resolve()

    requested = (project_root / candidate).resolve()
    if requested.exists():
        return requested

    # 2) Search by filename under Databases/databases
    db_roots_raw = [project_root / "Databases", project_root / "databases"]
    db_roots = dedupe_paths_case_insensitive([p for p in db_roots_raw if p.exists()])

    all_csvs = []
    exact_matches = []
    for db_root in db_roots:
        for csv_path in db_root.rglob("*.csv"):
            resolved_csv = csv_path.resolve()
            all_csvs.append(resolved_csv)
            if csv_path.name.lower() == candidate.name.lower():
                exact_matches.append(resolved_csv)

    all_csvs = dedupe_paths_case_insensitive(all_csvs)
    exact_matches = dedupe_paths_case_insensitive(exact_matches)

    if len(exact_matches) == 1:
        return exact_matches[0]
    if len(exact_matches) > 1:
        shown = "\n".join(str(p) for p in exact_matches[:10])
        raise FileNotFoundError(
            "Multiple datasets matched your filename. Use a more specific DATASET_RELATIVE path.\n"
            f"Matches:\n{shown}"
        )

    # 3) Partial-name fallback (for inputs like vapor.csv -> Hvapor.csv)
    token = candidate.stem.lower()
    partial_matches = [p for p in all_csvs if token and token in p.stem.lower()]
    partial_matches = dedupe_paths_case_insensitive(partial_matches)

    if len(partial_matches) == 1:
        print(f"Using partial match for dataset input '{dataset_input}': {partial_matches[0]}")
        return partial_matches[0]
    if len(partial_matches) > 1:
        shown = "\n".join(str(p) for p in partial_matches[:10])
        raise FileNotFoundError(
            "Multiple partial matches found. Use a more specific DATASET_RELATIVE path.\n"
            f"Matches:\n{shown}"
        )

    # 4) Suggest nearest filenames
    all_names = sorted({p.name for p in all_csvs})
    suggestions = get_close_matches(candidate.name, all_names, n=8, cutoff=0.3)
    suggestion_text = "\n".join(suggestions) if suggestions else "(no close filename suggestions)"

    alt_requested = (project_root / str(candidate).replace("Databases/", "databases/")).resolve()
    raise FileNotFoundError(
        f"Dataset not found. Tried: {requested}, {alt_requested}, and search under Databases/.\n"
        f"Closest filenames:\n{suggestion_text}\n"
        "Update DATASET_RELATIVE in this cell."
    )

# Update these paths/flags before each run
PROJECT_ROOT = resolve_project_root()
DATASET_RELATIVE = "Hvapor.csv"
DATASET_CSV = resolve_dataset_path(PROJECT_ROOT, DATASET_RELATIVE)
RUNS_ROOT = PROJECT_ROOT / "agent" / "run_archive"
OUTPUT_DIR_NAMES = ["CURATE", "GENERATE", "VERIFY", "PREDICT"]

# Command display and execution behavior
COMMAND_OPTION_ORDER = ["names", "y"]
COMMAND_CSV_NAME = DATASET_CSV.name  # display/copy-paste friendly
EXECUTION_CSV_NAME = str(DATASET_CSV)  # robust path for subprocess execution

# ROBERT command options are built from DATASET_CSV automatically.
ROBERT_OPTIONS = {
    "names": "Name",
    "y": "Hvapor",
    # "ignore": "[Name]",
    # "type": "reg",
    # "model": "[RF,GB,NN,MVL]",
    # "csv_test": "path/to/external_test.csv",
    # "kfold": 5,
    # "seed": 0,
}

RUN_ROBERT = True  # Set to False to skip execution and only print the resolved paths and options
COPY_INSTEAD_OF_MOVE = False

print("Resolved project root:", PROJECT_ROOT)
print("Resolved dataset:", DATASET_CSV)

Resolved project root: /Users/cjcscha/ROBERT/helper_rob/robert
Resolved dataset: /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv


## Cell 4 Guide: Preview Dataset Columns and Sample Rows

Use this cell to inspect the selected dataset before running ROBERT.

It helps you identify:
- likely molecule/name columns,
- likely target (y) columns,
- candidate descriptor/feature (x) columns.

After reviewing the output, return to Cell 2 and update `ROBERT_OPTIONS` (`names`, `y`, and `ignore`).

In [22]:
import pandas as pd

preview_rows = 8
df_preview = pd.read_csv(DATASET_CSV, nrows=preview_rows)

print(f"Dataset file: {DATASET_CSV}")
print(f"Preview rows loaded: {len(df_preview)}")
print(f"Column count: {len(df_preview.columns)}")

columns = list(df_preview.columns)
print("\nColumns:")
for idx, col in enumerate(columns, start=1):
    print(f"{idx:>3}. {col}")

name_hint_keywords = ["name", "mol", "molecule", "code", "id", "smiles"]
target_hint_keywords = ["target", "y", "yield", "ee", "ddg", "dg", "barrier", "tof", "activity", "log", "k", "outcome", "class"]

name_hints = [c for c in columns if any(k in c.lower() for k in name_hint_keywords)]
target_hints = [c for c in columns if any(k in c.lower() for k in target_hint_keywords)]

print("\nLikely name/molecule columns:", name_hints if name_hints else "none detected")
print("Likely target (y) columns:", target_hints if target_hints else "none detected")

ignore_suggestions = []
for col in name_hints:
    if col not in ignore_suggestions:
        ignore_suggestions.append(col)
if "smiles" in [c.lower() for c in columns]:
    smiles_exact = [c for c in columns if c.lower() == "smiles"]
    for col in smiles_exact:
        if col not in ignore_suggestions:
            ignore_suggestions.append(col)

feature_candidates = [c for c in columns if c not in set(ignore_suggestions + target_hints)]
print("\nCandidate ignore columns:", ignore_suggestions if ignore_suggestions else "review manually")
print(f"Candidate X-feature count after quick filtering: {len(feature_candidates)}")

print("\nTop preview rows:")
display(df_preview.head(preview_rows))

print("\nNext step:")
print("Update ROBERT_OPTIONS in Cell 2 using the inspected column names.")

Dataset file: /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv
Preview rows loaded: 8
Column count: 7

Columns:
  1. Name
  2. IF
  3. g3
  4. SS
  5. SGBP
  6. Hardness
  7. Hvapor

Likely name/molecule columns: ['Name']
Likely target (y) columns: none detected

Candidate ignore columns: ['Name']
Candidate X-feature count after quick filtering: 6

Top preview rows:


,Name,IF,g3,SS,SGBP,Hardness,Hvapor
0,3-Chlorobiphenyl,5155.712182,169.020579,69.91409,637.09491,0.308859,74.3
1,4-Chlorobiphenyl,5155.573881,148.191905,69.81467,637.06127,0.289335,71.6
2,"2,2prime-Dichlorobiphenyl",7335.521412,164.192584,55.26550,740.60900,0.335502,72.6
3,"2,3-Dichlorobiphenyl",7335.438844,164.641587,55.21398,740.59158,0.318685,73.4
4,"2,3prime-Dichlorobiphenyl",7335.655698,167.877833,55.29424,740.63454,0.315586,77.8
5,"2,4-Dichlorobiphenyl",7335.641197,165.649619,55.28842,740.63601,0.311422,75.4
6,"2,4prime-Dichlorobiphenyl",7335.640527,165.975291,55.28626,740.63179,0.310096,76.5
7,"3,3prime-Dichlorobiphenyl",7335.714669,152.904723,55.32234,740.65414,0.304095,81.0



Next step:
Update ROBERT_OPTIONS in Cell 2 using the inspected column names.


## Cell 6 Guide: Helper Functions

This cell defines utility functions used by the run workflow.

What these functions do:
- create a unique run folder using timestamp + dataset name,
- convert `ROBERT_OPTIONS` into command-line flags,
- build the ROBERT command from the selected dataset,
- run ROBERT only when requested,
- archive module output folders,
- write a run manifest JSON for traceability.

You normally do not edit this cell unless you want to change workflow behavior.

In [23]:
def sanitize_name(text: str) -> str:
    cleaned = []
    for ch in text:
        if ch.isalnum() or ch in ['-', '_']:
            cleaned.append(ch)
        else:
            cleaned.append('_')
    return ''.join(cleaned).strip('_') or 'dataset'


def build_run_folder(runs_root: Path, dataset_csv: Path) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dataset_tag = sanitize_name(dataset_csv.stem)
    run_folder = runs_root / f"{timestamp}__{dataset_tag}"
    run_folder.mkdir(parents=True, exist_ok=False)
    return run_folder


def options_to_cli_args(options: dict, preferred_order: list[str] | None = None) -> list[str]:
    args = []
    ordered_keys = []
    seen = set()

    if preferred_order:
        for key in preferred_order:
            if key in options:
                ordered_keys.append(key)
                seen.add(key)

    for key in options:
        if key not in seen:
            ordered_keys.append(key)

    for key in ordered_keys:
        value = options[key]
        flag = f"--{key}"
        if isinstance(value, bool):
            if value:
                args.append(flag)
            continue
        if value is None:
            continue
        args.extend([flag, str(value)])
    return args


def build_robert_command(csv_name_arg: str, options: dict, preferred_order: list[str] | None = None) -> list[str]:
    command = ["python", "-m", "robert"]
    command.extend(options_to_cli_args(options, preferred_order=preferred_order))
    command.extend(["--csv_name", str(csv_name_arg)])
    return command


def run_robert_if_requested(command: list[str], project_root: Path, enabled: bool) -> int | None:
    if not enabled:
        print("RUN_ROBERT is False: skipping ROBERT execution and archiving existing output folders only.")
        return None

    print("Running ROBERT command:")
    print(' '.join(command))
    result = subprocess.run(command, cwd=project_root, check=False)
    print(f"ROBERT exit code: {result.returncode}")
    return result.returncode


def archive_output_dirs(project_root: Path, run_folder: Path, output_dir_names: list[str], copy_only: bool) -> list[str]:
    archive_root = run_folder / "outputs"
    archive_root.mkdir(parents=True, exist_ok=True)

    archived = []
    for name in output_dir_names:
        src = project_root / name
        if not src.exists() or not src.is_dir():
            continue

        dest = archive_root / name
        if dest.exists():
            suffix = datetime.now().strftime("%H%M%S")
            dest = archive_root / f"{name}_{suffix}"

        if copy_only:
            shutil.copytree(src, dest)
        else:
            shutil.move(str(src), str(dest))

        archived.append(name)

    return archived


def write_manifest(run_folder: Path, project_root: Path, dataset_csv: Path, archived_dirs: list[str],
                   command: list[str], run_robert: bool, return_code: int | None, copy_only: bool) -> Path:
    manifest = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "project_root": str(project_root.resolve()),
        "dataset_csv": str(dataset_csv.resolve()),
        "run_robert": run_robert,
        "robert_command": command,
        "robert_return_code": return_code,
        "copy_instead_of_move": copy_only,
        "archived_output_dirs": archived_dirs,
        "expected_output_dirs": OUTPUT_DIR_NAMES,
    }

    manifest_path = run_folder / "run_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest_path

## Cell 7 Guide: Execute and Archive the Run

This cell performs the workflow in order.

It will:
1. verify that the selected dataset file exists,
2. build and print the final ROBERT command,
3. optionally run ROBERT,
4. move or copy `CURATE`, `GENERATE`, `VERIFY`, and `PREDICT` into a unique archive folder,
5. write `run_manifest.json` with metadata for reproducibility.

After execution, check the printed run folder path and manifest path to confirm success.

In [26]:
if not DATASET_CSV.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_CSV}. "
        "Re-run Cell 3 and verify DATASET_RELATIVE points to a real file under Databases/."
    )

ROBERT_COMMAND_DISPLAY = build_robert_command(
    COMMAND_CSV_NAME, ROBERT_OPTIONS, preferred_order=COMMAND_OPTION_ORDER
    )
ROBERT_COMMAND = build_robert_command(
    EXECUTION_CSV_NAME, ROBERT_OPTIONS, preferred_order=COMMAND_OPTION_ORDER
    )

print("Resolved ROBERT command (copy/paste style):", ' '.join(ROBERT_COMMAND_DISPLAY))
print("Execution ROBERT command:", ' '.join(ROBERT_COMMAND))

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
run_folder = build_run_folder(RUNS_ROOT, DATASET_CSV)

return_code = run_robert_if_requested(ROBERT_COMMAND, PROJECT_ROOT, RUN_ROBERT)
archived_dirs = archive_output_dirs(PROJECT_ROOT, run_folder, OUTPUT_DIR_NAMES, COPY_INSTEAD_OF_MOVE)
manifest_path = write_manifest(
    run_folder=run_folder,
    project_root=PROJECT_ROOT,
    dataset_csv=DATASET_CSV,
    archived_dirs=archived_dirs,
    command=ROBERT_COMMAND,
    run_robert=RUN_ROBERT,
    return_code=return_code,
    copy_only=COPY_INSTEAD_OF_MOVE,
)

print("Run folder:", run_folder)
print("Archived output dirs:", archived_dirs if archived_dirs else "none found")
print("Manifest:", manifest_path)

Resolved ROBERT command (copy/paste style): python -m robert --names Name --y Hvapor --csv_name Hvapor.csv
Execution ROBERT command: python -m robert --names Name --y Hvapor --csv_name /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv
Running ROBERT command:
python -m robert --names Name --y Hvapor --csv_name /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv
ROBERT v 2.1.0 2026/05/13 16:08:19 
How to cite: Dalmau, D.; Alegre Requena, J. V. WIREs Comput Mol Sci. 2024, 14, e1733.


Command line used in ROBERT: python -m robert --names "Name" --y "Hvapor" --csv_name "/Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/Hvapor.csv"



o  Starting data curation with the CURATE module


o  Database Hvapor.csv loaded successfully, including:
   - 133 datapoints
   - 5 accepted descriptors
   - 1 ignored descriptors
   - 0 discarded descriptors


o  Analyzing categorical variables
   - No categorical variables were found


o  Duplication filt

## Usage Notes

1. In Cell 2, set `DATASET_RELATIVE` to your dataset path or filename (example: `vapor.csv`).
2. In Cell 4, inspect headers and sample rows to identify `names`, `y`, and likely `ignore` columns.
3. Back in Cell 2, update `ROBERT_OPTIONS` to build the command without manual CLI editing.
4. In Cell 8, confirm the printed resolved command looks correct.
5. Set `RUN_ROBERT = True` in Cell 2 when ready to execute.
6. If you want to keep generic output folders in place, set `COPY_INSTEAD_OF_MOVE = True`.

Each run is archived in `agent/run_archive/<timestamp>__<dataset_name>/`.